In [20]:
import pandas as pd
import numpy as np
import sys, os
from importlib import reload
import yaml
from pathlib import Path
from dataclasses import asdict

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from analysis import plots
import analysis.report
import training.gbm_model_trainer
from training.gbm_model_trainer import GBMModelTrainerConfig 

reload(analysis)
reload(analysis.plots)
reload(analysis.report)
reload(training.gbm_model_trainer)

<module 'training.gbm_model_trainer' from 'c:\\Users\\ASacco\\OneDrive - Plymouth Rock Assurance Corp\\repos\\analysis-tools\\src\\training\\gbm_model_trainer.py'>

## Load data

In [3]:
df = pd.read_csv("../data/bank-full.csv", delimiter=';')

In [4]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [5]:
# Convert 'y' to numeric for analysis
df["target"] = np.where(df["y"] == "no", 0, 1)

# Add weights to test weight parameter across functions 
df["weights"] = np.abs(np.random.randn(len(df.index)))

# Add an arbitrary data split for testing functions (TVH = 40/30/30)
df["random"] = np.random.uniform(0, 1, len(df.index))
df["split"] = np.where(
    df["random"]  > 0.70, 
    "V",
    np.where(
        df["random"]  > 0.40, 
        "H", 
        "T"
    )
)

In [6]:
df["split"].value_counts(dropna=False, normalize=True)

split
T    0.401805
V    0.299927
H    0.298268
Name: proportion, dtype: float64

In [7]:
df.groupby("split")["target"].mean()

split
H    0.117316
T    0.113454
V    0.121386
Name: target, dtype: float64

In [9]:
# Split data; define features and target
train = df.query("split == 'T'").drop(columns=["y", "split"])
test = df.query("split == 'V'").drop(columns=["y", "split"])
holdout = df.query("split == 'H'").drop(columns=["y", "split"])

## Use ModelTrainer class for training
- Can use example config files in `analysis-tools/examples` or create a config within your notebook and save it locally

In [42]:
# Define your config
config_xgb = GBMModelTrainerConfig(

    # Training parameters
    actual_col="target",
    predicted_col="pred_xgb",

    hyperparameters={
        "objective": "binary:logistic",
        "n_estimators": 100,
        "max_depth": 3,
        "learning_rate": 0.1,
        "subsample": 0.5,
        "colsample_bytree": 0.5,
        "random_state": 42,
        "early_stopping": 25,
    },

    # Logging parameters
    output_log=True,
    output_report=True,
    log_file="outputs/training_xgb.log",

    # Reporting parameters
    report_file="outputs/model_analysis.html",
    report_params={
    },
    tabulate_vars=["job", "marital", "education"],
    plots_to_add=[
        {
            "plot": "plot_error_by_group_grid",
            "title": "Error by Analysis Variables",
            "kwargs": {
                "group_cols": ["job", "education", "age"]
            }
        },
        {
            "plot": "gain_curve_with_gini",
            "title": "Gain Curve / Lorenz Curve"
        },
        {
            "plot": "partial_gini_plot",
            "title": "Partial Gini (Top 15%)",
            "kwargs": {
                "top_percent": 15
            }
        },
        {
            "plot": "lift_chart",
            "title": "Lift Chart"
        },
        {
            "plot": "crunched_residual_plot",
            "title": "Crunched Residuals"
        },
        {
            "plot": "plot_residual_fit",
            "title": "Std and Avg of Normalized Residuals",
            "kwargs": {
                "residual_type": "normalized"
            }
        }
    ],
)

# Output config to JSON
config_output_path = Path("../examples/slim_training_config.yaml")

import yaml
# Write config to JSON file
with config_output_path.open("w") as f:
    yaml.dump(asdict(config_xgb), f, indent=2)

print(f"Config saved to {config_output_path.resolve()}")

Config saved to C:\Users\ASacco\OneDrive - Plymouth Rock Assurance Corp\repos\analysis-tools\examples\slim_training_config.yaml


In [52]:
reload(training.gbm_model_trainer)
reload(analysis.report)
reload(analysis.plots)

<module 'analysis.plots' from 'c:\\Users\\ASacco\\OneDrive - Plymouth Rock Assurance Corp\\repos\\analysis-tools\\src\\analysis\\plots.py'>

In [53]:
from training.gbm_model_trainer import GBMModelTrainer
from xgboost import XGBClassifier

mt_xgboost = GBMModelTrainer(
    model_class=XGBClassifier,
    config_path="../examples/slim_training_config.yaml",
    train_df=train,
    valid_df=test,
    holdout_df=holdout
)

In [54]:
# mt_xgboost.config
help(XGBClassifier.fit)

Help on function fit in module xgboost.sklearn:

fit(
    self,
    X: Any,
    y: Any,
    *,
    sample_weight: Optional[Any] = None,
    base_margin: Optional[Any] = None,
    eval_set: Optional[Sequence[Tuple[Any, Any]]] = None,
    verbose: Union[bool, int, NoneType] = True,
    xgb_model: Union[xgboost.core.Booster, str, xgboost.sklearn.XGBModel, NoneType] = None,
    sample_weight_eval_set: Optional[Sequence[Any]] = None,
    base_margin_eval_set: Optional[Sequence[Any]] = None,
    feature_weights: Optional[Any] = None
) -> 'XGBClassifier'
    Fit gradient boosting classifier.

    Note that calling ``fit()`` multiple times will cause the model object to be
    re-fit from scratch. To resume training from a previous checkpoint, explicitly
    pass ``xgb_model`` argument.

    Parameters
    ----------
    X :
        Input feature matrix. See :ref:`py-data` for a list of supported types.

        When the ``tree_method`` is set to ``hist``, internally, the
        :py:class:`Qu

In [55]:
mt_xgboost.train()

TypeError: XGBClassifier.fit() got an unexpected keyword argument 'callbacks'

In [25]:
mt_xgboost.log_lines

['[2025-07-30 15:55:24] Instantiated model: XGBClassifier',
 '[2025-07-30 15:55:24] Training completed in 0.1600 seconds',
 '[2025-07-30 15:55:24] Model: XGBClassifier',
 "[2025-07-30 15:55:24] Hyperparameters: {'colsample_bytree': 0.5, 'early_stopping': 25, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'objective': 'binary:logistic', 'random_state': 42, 'subsample': 0.5}",
 '[2025-07-30 15:55:24] Training data shape: (18166, 19)',
 '[2025-07-30 15:55:24] Validation data shape: (13560, 19)',
 '[2025-07-30 15:55:24] Number of predictors: 18',
 "[2025-07-30 15:55:24] Target column: 'target'",
 "[2025-07-30 15:55:24] Prediction column: 'pred_xgb'",
 '[2025-07-30 15:55:24] Validation score (default metric): 0.9044',
 '[2025-07-30 15:55:25] ✔️ Added plot: Error by Analysis Variables (plot_error_by_group_grid)',
 '[2025-07-30 15:55:26] ✔️ Added plot: Gain Curve / Lorenz Curve (gain_curve_with_gini)',
 '[2025-07-30 15:55:26] ✔️ Added plot: Partial Gini (Top 15%) (partial_gini_plo

## Test with CAT Boost

In [35]:
from training.gbm_model_trainer import GBMModelTrainer
from catboost import CatBoostClassifier
reload(training.gbm_model_trainer)

<module 'training.gbm_model_trainer' from 'c:\\Users\\ASacco\\OneDrive - Plymouth Rock Assurance Corp\\repos\\analysis-tools\\src\\training\\gbm_model_trainer.py'>

In [36]:
# Define your config
config_catboost = GBMModelTrainerConfig(

    # Training parameters
    actual_col="target",
    predicted_col="pred_xgb",

    hyperparameters={
        "loss_function": "Logloss", # Valid for CATBoost
        "eval_metric": "NormalizedGini",
        "n_estimators": 300,
        "max_depth": 8,
        "learning_rate": 0.1,
        "subsample": 0.5,
        "colsample_bytree": 0.5,
        "random_state": 42,
    },

    # Logging parameters
    output_log=True,
    output_report=True,
    log_file="training_catboost.log",

    # Reporting parameters
    report_file="model_analysis_catboost.html",
    report_params={
    },
    tabulate_vars=["job", "marital", "education"],
    plots_to_add=[
        {
            "plot": "plot_error_by_group_grid",
            "title": "Error by Analysis Variables",
            "kwargs": {
                "group_cols": ["job", "education", "age"]
            }
        },
        {
            "plot": "gain_curve_with_gini",
            "title": "Gain Curve / Lorenz Curve"
        },
        {
            "plot": "partial_gini_plot",
            "title": "Partial Gini (Top 15%)",
            "kwargs": {
                "top_percent": 15
            }
        },
        {
            "plot": "lift_chart",
            "title": "Lift Chart"
        },
        {
            "plot": "crunched_residual_plot",
            "title": "Crunched Residuals"
        },
        {
            "plot": "plot_residual_fit",
            "title": "Std and Avg of Normalized Residuals",
            "kwargs": {
                "residual_type": "normalized"
            }
        }
    ],
)

# Output config to JSON
config_output_path = Path("../examples/model_config_catboost.yaml")

# Write config to JSON file
with config_output_path.open("w") as f:
    yaml.dump(asdict(config_catboost), f, indent=2)

print(f"Config saved to {config_output_path.resolve()}")

Config saved to C:\Users\ASacco\OneDrive - Plymouth Rock Assurance Corp\repos\analysis-tools\examples\model_config_catboost.yaml


In [37]:
mt_catboost = GBMModelTrainer(
    model_class=CatBoostClassifier,
    config_path="../examples/model_config_catboost.yaml",
    train_df=train,
    valid_df=test
)

In [38]:
mt_catboost.config

GBMModelTrainerConfig(actual_col='target', predicted_col='pred_xgb', output_dir='outputs', log_file='training_catboost.log', report_file='model_analysis_catboost.html', hyperparameters={'colsample_bytree': 0.5, 'eval_metric': 'NormalizedGini', 'learning_rate': 0.1, 'loss_function': 'Logloss', 'max_depth': 8, 'n_estimators': 300, 'random_state': 42, 'subsample': 0.5}, output_log=True, output_report=True, report_params={}, plots_to_add=[{'kwargs': {'group_cols': ['job', 'education', 'age']}, 'plot': 'plot_error_by_group_grid', 'title': 'Error by Analysis Variables'}, {'plot': 'gain_curve_with_gini', 'title': 'Gain Curve / Lorenz Curve'}, {'kwargs': {'top_percent': 15}, 'plot': 'partial_gini_plot', 'title': 'Partial Gini (Top 15%)'}, {'plot': 'lift_chart', 'title': 'Lift Chart'}, {'plot': 'crunched_residual_plot', 'title': 'Crunched Residuals'}, {'kwargs': {'residual_type': 'normalized'}, 'plot': 'plot_residual_fit', 'title': 'Std and Avg of Normalized Residuals'}], tabulate_vars=['job', 

In [39]:
mt_catboost.train()

0:	test: 0.6182176	best: 0.6182176 (0)	total: 65.4ms	remaining: 19.5s
50:	test: 0.8524568	best: 0.8524568 (50)	total: 3.4s	remaining: 16.6s
100:	test: 0.8584596	best: 0.8591561 (88)	total: 6.86s	remaining: 13.5s
150:	test: 0.8618815	best: 0.8623163 (134)	total: 10.5s	remaining: 10.4s
Stopped by overfitting detector  (30 iterations wait)

bestTest = 0.8627490535
bestIteration = 166

Shrink model to first 167 iterations.


c:\Users\ASacco\OneDrive - Plymouth Rock Assurance Corp\repos\analysis-tools\src\analysis\plots.py:636: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
c:\Users\ASacco\OneDrive - Plymouth Rock Assurance Corp\repos\analysis-tools\src\analysis\plots.py:636: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


✅ Analysis report generated at outputs\model_analysis_catboost.html


In [41]:
mt_catboost.model.best_score_

{'learn': {'Logloss': 0.1231360762864535},
 'validation': {'NormalizedGini': 0.8627490535145457,
  'Logloss': 0.20754535434340332}}

## Test with LightGBM

In [234]:
# Define your config
config_lgbm = GBMModelTrainerConfig(

    # Training parameters
    actual_col="target",
    predicted_col="pred_xgb",

    hyperparameters={
        # "objective": "binary:logistic",
        "loss_function": "Logloss", # Valid for LightGBM
        "n_estimators": 100,
        "max_depth": 3,
        "learning_rate": 0.1,
        "subsample": 0.5,
        "colsample_bytree": 0.5,
        "random_state": 42,
        "early_stopping": 25,
    },

    # Logging parameters
    output_log=True,
    output_report=True,
    output_email=True,
    email="asacco@plymouthrock.com",
    log_file="outputs/training_lgbm.log",

    # Reporting parameters
    report_output_path="outputs/model_analysis_lgbm.html",
    report_params={
    },
    tabulate_vars=["job", "marital", "education"],
    plots_to_add=[
        {
            "plot": "plot_error_by_group_grid",
            "title": "Error by Analysis Variables",
            "kwargs": {
                "group_cols": ["job", "education", "age"]
            }
        },
        {
            "plot": "gain_curve_with_gini",
            "title": "Gain Curve / Lorenz Curve"
        },
        {
            "plot": "partial_gini_plot",
            "title": "Partial Gini (Top 15%)",
            "kwargs": {
                "top_percent": 15
            }
        },
        {
            "plot": "lift_chart",
            "title": "Lift Chart"
        },
        {
            "plot": "crunched_residual_plot",
            "title": "Crunched Residuals"
        },
        {
            "plot": "plot_residual_fit",
            "title": "Std and Avg of Normalized Residuals",
            "kwargs": {
                "residual_type": "normalized"
            }
        }
    ],
)

# Output config to YAML
config_output_path = Path("model_config_lgbm.yaml")

# Write config to YAML file
with config_output_path.open("w") as f:
    json.dump(asdict(config_lgbm), f, indent=2)

print(f"✅ Config saved to {config_output_path.resolve()}")

✅ Config saved to /home/sagemaker-user/analysis-tools/notebooks/model_config_lgbm.yaml


In [235]:
reload(training.gbm_model_trainer)
reload(analysis.report)
reload(analysis.plots)
from training.gbm_model_trainer import GBMModelTrainer
from lightgbm import LGBMClassifier

mt_lgbm = GBMModelTrainer(
    model_class=LGBMClassifier,
    config_path="model_config_lgbm.yaml",
    train_df=train,
    valid_df=test
)

In [236]:
mt_lgbm.config

GBMModelTrainerConfig(actual_col='target', predicted_col='pred_xgb', log_file='outputs/training_lgbm.log', report_output_path='outputs/model_analysis_lgbm.html', email='asacco@plymouthrock.com', hyperparameters={'loss_function': 'Logloss', 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'subsample': 0.5, 'colsample_bytree': 0.5, 'random_state': 42, 'early_stopping': 25}, output_log=True, output_email=True, output_report=True, report_params={}, plots_to_add=[{'plot': 'plot_error_by_group_grid', 'title': 'Error by Analysis Variables', 'kwargs': {'group_cols': ['job', 'education', 'age']}}, {'plot': 'gain_curve_with_gini', 'title': 'Gain Curve / Lorenz Curve'}, {'plot': 'partial_gini_plot', 'title': 'Partial Gini (Top 15%)', 'kwargs': {'top_percent': 15}}, {'plot': 'lift_chart', 'title': 'Lift Chart'}, {'plot': 'crunched_residual_plot', 'title': 'Crunched Residuals'}, {'plot': 'plot_residual_fit', 'title': 'Std and Avg of Normalized Residuals', 'kwargs': {'residual_type': 'norm

In [237]:
mt_lgbm.train()

[LightGBM] [Info] Number of positive: 3736, number of negative: 27835
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001115 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1503
[LightGBM] [Info] Number of data points in the train set: 31571, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.118336 -> initscore=-2.008279
[LightGBM] [Info] Start training from score -2.008279
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-packages/seaborn/_oldcore.py:1108: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context("mode.use_inf_as_na", True):
/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-packages/seaborn/_oldcore.py:1108: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context("mode.use_inf_as_na", True):
/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-packages/seaborn/_oldcore.py:1108: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context("mode.use_inf_as_na", True):
/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-

✅ Analysis report generated at outputs/model_analysis_lgbm.html


In [238]:
mt_lgbm.log_lines

["[2025-07-28 16:14:44] ⚠️ 'loss_function' is not a valid hyperparameter for LGBMClassifier and will be ignored. ",
 "[2025-07-28 16:14:44] ⚠️ 'early_stopping' is not a valid hyperparameter for LGBMClassifier and will be ignored. ",
 "[2025-07-28 16:14:44] ⚠️ 'verbose' is not a valid hyperparameter for LGBMClassifier and will be ignored. ",
 '[2025-07-28 16:14:44] Instantiated model: LGBMClassifier',
 '[2025-07-28 16:14:45] Training completed in 0.1457 seconds',
 '[2025-07-28 16:14:45] Model: LGBMClassifier',
 "[2025-07-28 16:14:45] Hyperparameters: {'loss_function': 'Logloss', 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'subsample': 0.5, 'colsample_bytree': 0.5, 'random_state': 42, 'early_stopping': 25}",
 '[2025-07-28 16:14:45] Training data shape: (31571, 19)',
 '[2025-07-28 16:14:45] Validation data shape: (13640, 19)',
 '[2025-07-28 16:14:45] Number of predictors: 18',
 "[2025-07-28 16:14:45] Target column: 'target'",
 "[2025-07-28 16:14:45] Prediction column: 'pred